In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

<Accordion>
<AccordionItem title="Versões dos pacotes">

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou versões mais recentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

A primitiva Executor faz parte do [modelo de execução direcionada](/guides/directed-execution-model), que fornece mais flexibilidade ao personalizar um fluxo de trabalho de mitigação de erros.

As entradas e saídas da primitiva Executor são muito diferentes das primitivas [Sampler](/guides/sampler-input-output) e [Estimator](/guides/estimator-input-output). Por exemplo, em vez de receber uma lista de PUBs como entrada, o Executor recebe um `QuantumProgram`, que contém uma lista de objetos `QuantumProgramItem`. Essas classes de contêiner oferecem mais flexibilidade do que um PUB, que é uma estrutura de dados de tupla simples.

A saída do Executor é um `QuantumProgramResult`, que é iterável e contém um elemento para cada `QuantumProgramItem` de entrada.

<span id="programs"></span>

## Entradas: Programas quânticos

Como dito anteriormente, a entrada para uma primitiva Executor é um [`QuantumProgram`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgram.html#QuantumProgram), que é um iterável de objetos [`QuantumProgramItem`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgramItem.html). Esses objetos podem ser de dois tipos:
- `CircuitItem`, que normalmente armazena um Circuit e seus valores de parâmetros (se houver).
- `SamplexItem`, que normalmente armazena o seguinte:
    - Um Circuit modelo
    - Um objeto samplex, que é usado para gerar conjuntos randomizados de parâmetros em tempo de execução (por exemplo, para realizar twirling ou injetar ruído)
    - Argumentos para o samplex, que podem incluir valores de parâmetros para o Circuit original

Cada um desses itens representa uma tarefa diferente para o Executor realizar.

### Antes de começar

Alguns dos exemplos de código nesta página usam `samplex`, que faz parte do pacote Samplomatic. Portanto, antes de executar esses blocos de código, você deve instalar o Samplomatic, conforme mostrado no bloco de código a seguir. Para mais informações, veja a [documentação do Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit_ibm_runtime import Executor, QiskitRuntimeService
from qiskit.circuit import Parameter, QuantumCircuit
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Initialize an empty program
program = QuantumProgram(shots=1024)

# Initialize and transpile a 3-qubit quantum circuit with 2 parameters.
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.rz(Parameter("theta"), 0)
circuit.rz(Parameter("phi"), 1)

# `measure_all` adds a 3-bit classical register named "meas"
circuit.measure_all()

# Choose the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Generate a preset pass manager
# This will be used to convert the abstract circuit to an
# equivalent Instruction Set Architecture (ISA) circuit.
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Transpile the circuit
isa_circuit = preset_pass_manager.run(circuit)

### Exemplo: Criar um `QuantumProgram` com duas tarefas diferentes
Primeiro inicialize seu programa quântico, depois acrescente itens de programa a ele usando `append_circuit_item` ou `append_samplex_item` (se um samplex estiver presente), conforme mostrado nos exemplos a seguir.

A célula a seguir inicializa um `QuantumProgram` e especifica que ele deve executar 1024 shots para cada configuração de cada item no programa.

> **Note:** Diferentemente do Sampler, um `QuantumProgram` aceita apenas um único valor de shots. Se você quiser um valor de shots diferente, precisará de um `QuantumProgram` separado, que seria um job separado.

In [2]:
# Append the transpiled circuit and an array
# containing 10 sets of parameter values to the program
program.append_circuit_item(
    isa_circuit,
    circuit_arguments=np.random.rand(
        10, 2
    ),  # 10 sets of parameter values and 2 parameters
)

### Acrescentar um `CircuitItem`
Em seguida, acrescente o Circuit alvo, que foi transpilado de acordo com a arquitetura de conjunto de instruções (ISA) do Backend, ao `QuantumProgram`. Como este Circuit tem dois parâmetros, também devemos fornecer os valores dos parâmetros (10 conjuntos neste exemplo). Executar este `CircuitItem` é a primeira tarefa que o programa realizará.

In [3]:
# Transpile the circuit, additionally grouping gates and measurements into annotated boxes
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Use the boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)
boxed_circuit = preset_pass_manager.run(circuit)

# Build the template circuit and the samplex.  The template circuit has parametric gates
# without fixed values and the samplex randomly generates the parameter
# values on the server side at runtime to perform twirling.
template_circuit, samplex = build(boxed_circuit)

# Determine what arguments are required by the samplex.
# Input the arguments in samplex_arguments.
print(samplex.inputs())

TensorInterface(<
  - 'parameter_values' <float64[2]>: Input parameter values to use during sampling.
>)


In [4]:
# Append the template circuit and samplex as a samplex item
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    samplex_arguments={
        # the arguments required by the samplex.sample method
        "parameter_values": np.random.rand(10, 2),
    },
    shape=(28, 10),  # 28 randomizations and 10 sets of parameter values
)

In [5]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)

# Retrieve the result
result = job.result()

## Outputs

Executor's output is a [`QuantumProgramResult`](/docs/api/qiskit-ibm-runtime/results-quantum-program-result), which is an iterable. It contains one entry per input `QuantumProgramItem` in the same order as the input items. Each of these output items is a dictionary where the keys are strings that correspond the classical registers' names in the input circuits (among others), so you no longer need to memorize these names like you did with Sampler output. The dictionary values are of type `np.ndarray`.

The result for the previous example contains these items:

### `CircuitItem` result

The first item contains the results of running the first task (a `CircuitItem`) in the program. It contains a single key, `meas`, which is the name of the classical register in the input circuit. The value of this key maps to an `np.ndarray` of shape `(parameter sets, shots, register bits)`, which is (10, 1024, 3) for the above example.

The following code illustrates how to access this information:

In [6]:
# Access the results of the classical register of task #0, a CircuitItem
result_0 = result[0]["meas"]
print(f"Result shape: {result_0.shape}")

Result shape: (10, 1024, 3)


### `SamplexItem` result

The second item contains the results of running the second task (a `SamplexItem`) in the program. This item contains multiple keys. The `meas` key, which is the name of the input circuit's classical register, maps to that register's array of results. This array has the shape `(randomizations, parameter sets, shots, classical bits)`, or (28, 10, 1024, 3) in this example. Additionally, the output contains a `measurement_flips.meas` key, which is the bit-flip corrections to undo the measurement twirling for the `meas` register.  This output shape will be (28, 10, 1, 3) for our example because only one shot is required to perform the bit-flip.

In [7]:
# Access the results of the classical register of task #1
result_1 = result[1]["meas"]
print(f"Result shape: {result_1.shape}")

# Access the bit-flip corrections
flips_1 = result[1]["measurement_flips.meas"]
print(f"Bit-flip corrections shape: {flips_1.shape}")

# Undo the bit flips via classical XOR
unflipped_result_1 = result_1 ^ flips_1

Result shape: (28, 10, 1024, 3)
Bit-flip corrections shape: (28, 10, 1, 3)


## Saídas
A saída do Executor é um [`QuantumProgramResult`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgramResult.html#QuantumProgramResult), que é iterável. Ele contém uma entrada por `QuantumProgramItem` de entrada na mesma ordem dos itens de entrada. Cada um desses itens de saída é um dicionário onde as chaves são strings que correspondem aos nomes dos registradores clássicos nos Circuits de entrada (entre outros), portanto você não precisa mais memorizar esses nomes como fazia com a saída do Sampler. Os valores do dicionário são do tipo `np.ndarray`.

O resultado para o exemplo anterior contém estes itens:

### Resultado do `CircuitItem`
O primeiro item contém os resultados da execução da primeira tarefa (um `CircuitItem`) no programa. Ele contém uma única chave, `meas`, que é o nome do registrador clássico no Circuit de entrada. O valor desta chave mapeia para um `np.ndarray` de formato `(conjuntos de parâmetros, shots, bits do registrador)`, que é (10, 1024, 3) para o exemplo acima.

O código a seguir ilustra como acessar estas informações: